# Lesson 06 Lab — Profiling Mixed Precision and Verifying Dispatch

**Puzzle:** If autocast made an operation faster, does that prove the intended low-precision kernel ran?

This notebook keeps the RTX 5090 outputs from a complete run. Read the theory cells, make a prediction, and then use **Run All** on your own GPU.


## Why this matters

Timing and dispatch are different claims. A faster autocast region shows an application-level effect; a PyTorch profiler event identifies framework operators; only a lower-level trace can justify a native kernel or Tensor Core utilization claim. Good profiling keeps those evidence levels separate instead of using one as a shortcut for another.


## 0. Predict before running

1. Predict which PyTorch operator events should surround a BF16 matrix multiplication under autocast.
2. Explain why warm-up and synchronization are required before comparing CUDA timings.
3. Name the additional evidence needed to claim a particular native Tensor Core kernel.

For each answer, name the observation that would prove you wrong.


## 1. Name the concrete objects

Three evidence layers answer different questions: model outputs show semantic effect, framework operators show graph dispatch, and native kernel traces show the implementation actually launched.

- A wall-clock delta and an operator trace answer different questions.
- Warm-up removes initialization and compilation from the steady-state sample.
- PyTorch operator names are higher-level evidence than native kernel names; use Nsight when kernel identity matters.


## 2. Derive the mechanism

Profiling can expose casts, copies, GEMMs, launch count, and device time. Warm-up is required because lazy initialization, compilation, and allocator growth are not steady-state execution.

GPU launches are asynchronous: host elapsed time can measure queue submission rather than device completion. CUDA events timestamp work in the device stream, but initialization, allocator growth, lazy library loading, and compilation can still contaminate early samples. A defensible steady-state number therefore specifies warm-up, synchronization, sample count, and a distribution statistic.

A trace adds causality. At the framework layer, events such as `aten::matmul`, `aten::mm`, and casts reveal the operation graph and unexpected conversions. At the native layer, kernel names and hardware counters reveal tile implementation, tensor-pipe activity, occupancy, and bandwidth. The layers answer complementary questions; neither makes the other redundant.


## 3. Verify the execution environment

The next cell asserts CUDA availability, fixes the seed, locates the lesson, and prints a sanitized GPU/PyTorch/CUDA record. Check it before interpreting output.


In [1]:
from pathlib import Path
import json
import sys
import torch

chapter_rel = Path("chapters/01-mixed-precision-int4")
repo_root = next(
    p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / chapter_rel / "support" / "lab_common.py").exists()
)
sys.path.insert(0, str(repo_root / chapter_rel / "support"))
from lab_common import (base_result, cuda_benchmark, environment_record,
                        error_metrics, require_cuda, save_result,
                        symmetric_quantize)

lesson_dir = repo_root / chapter_rel / "06-mixed-precision-profiling"
device = require_cuda()
torch.manual_seed(2026 + 6)
environment = environment_record()
print(json.dumps(environment, indent=2))


{
  "gpu": "NVIDIA GeForce RTX 5090",
  "compute_capability": "12.0",
  "gpu_memory_gib": 31.358,
  "python": "3.12.13",
  "torch": "2.12.0",
  "cuda_runtime": "13.0"
}


## 4. Freeze the comparison

| Role | This run |
|---|---|
| Baseline | theoretical expectation that autocast selects BF16 for an eligible GEMM |
| Candidate | actual timed autocast region plus captured PyTorch operator events |
| Held constant | 2048×2048 shape, seed, GPU, five warm-ups, fifteen CUDA-event samples |
| Measurements | median/p90 latency and selected framework operator names |
| Evidence | `pytorch-gpu` |

**Experiment:** Profile an autocast BF16 GEMM with PyTorch Profiler and record the relevant operator events beside CUDA-event timing.


## 5. Read the experiment code

The notebook pairs repeated CUDA-event timing with selected PyTorch profiler events and explicitly stops short of inventing a native kernel name.

The notebook profiles one BF16 autocast matrix multiplication and records selected events from the PyTorch profiler. It separately times the same region with CUDA events. Keeping trace collection outside the timed samples avoids conflating profiler overhead with normal latency.

The result schema calls the events `pytorch_operator_events`, not `native_kernels`. That naming is deliberate: a framework trace is sufficient to audit the Python-level path but not to quantify Tensor Core occupancy.

Only after these variables match the protocol should the cell be executed.


In [2]:
from torch.profiler import ProfilerActivity, profile
import warnings
a = torch.randn(2048, 2048, device=device); b = torch.randn(2048, 2048, device=device)
def work():
    with torch.autocast("cuda", dtype=torch.bfloat16): return a @ b
timing = cuda_benchmark(work, warmup=5, repeats=15)
with warnings.catch_warnings():
    warnings.filterwarnings("ignore", message=".*Profiler clears events.*")
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for _ in range(3): work()
torch.cuda.synchronize()
events = []
for e in prof.key_averages():
    if any(k in e.key.lower() for k in ("mm", "matmul", "to", "copy")):
        events.append({"operator": e.key, "count": e.count})
result = base_result(6, "pytorch-gpu"); result.update({"shape": [2048, 2048], "timing": timing,
    "pytorch_operator_events": events[:20], "conclusion": "Autocast timing and PyTorch operator evidence were captured; native kernel identity was not claimed."})


## 6. Read the retained RTX 5090 result

**Recorded environment:** NVIDIA GeForce RTX 5090; compute capability 12.0; PyTorch 2.12.0; CUDA runtime 13.0.

| Measured field | Checked-in value |
|---|---:|
| GEMM shape | 2048 × 2048 |
| Median | 0.104416 ms |
| p90 | 0.106240 ms |
| Samples | 15 |
| PyTorch operator events | {'count': 6, 'operator': 'aten::matmul'}, {'count': 6, 'operator': 'aten::to'}, {'count': 6, 'operator': 'aten::_to_copy'}, {'count': 6, 'operator': 'aten::copy_'}, {'count': 3, 'operator': 'aten::mm'} |


## 7. Interpret rather than merely print

The saved run measured a 0.104416 ms median and 0.106240 ms p90 over fifteen samples. Five relevant PyTorch operator events were retained. The tight median-to-p90 spread suggests a stable microbenchmark after warm-up, while the event list confirms that an autocast/matmul path was captured.

Nothing in these two fields identifies one SASS kernel or reports hardware utilization. The bounded conclusion is therefore that the application path and timing were observed; a native dispatch claim remains open.

**Inspection rule:** Require both repeated timing and trace evidence. This lab deliberately labels PyTorch operators rather than claiming a native kernel name.


## 8. Keep the evidence label honest

This run is labeled **`pytorch-gpu`**. The measured tensors and operations ran on CUDA through PyTorch. The result does not name a separate production backend unless an operator trace identifies it.

The next cell writes the complete structured result; its existing saved output is part of the checked-in evidence.


In [3]:
artifact_path = save_result(result, lesson_dir)
print(json.dumps(result, indent=2, sort_keys=True))
print("Saved: artifacts/rtx5090-result.json")


{
  "conclusion": "Autocast timing and PyTorch operator evidence were captured; native kernel identity was not claimed.",
  "environment": {
    "compute_capability": "12.0",
    "cuda_runtime": "13.0",
    "gpu": "NVIDIA GeForce RTX 5090",
    "gpu_memory_gib": 31.358,
    "python": "3.12.13",
    "torch": "2.12.0"
  },
  "evidence_label": "pytorch-gpu",
  "executed_at_utc": "2026-08-07T14:49:58+00:00",
  "lesson": 6,
  "pytorch_operator_events": [
    {
      "count": 6,
      "operator": "aten::matmul"
    },
    {
      "count": 6,
      "operator": "aten::to"
    },
    {
      "count": 6,
      "operator": "aten::_to_copy"
    },
    {
      "count": 6,
      "operator": "aten::copy_"
    },
    {
      "count": 3,
      "operator": "aten::mm"
    }
  ],
  "schema_version": 1,
  "shape": [
    2048,
    2048
  ],
  "timing": {
    "median_ms": 0.104416,
    "p90_ms": 0.10624,
    "repeats": 15,
    "samples_ms": [
      0.115168,
      0.10736,
      0.105472,
      0.10624,
    

## 9. Make the bounded decision

> Use a two-part proof: controlled timing for effect and profiler evidence for dispatch; escalate to Nsight for native-kernel claims.

**Acceptance/rollback:** First reproduce timing without a profiler, then capture a short aligned trace. Name only the level actually observed: PyTorch operator, CUDA kernel, or end-to-end phase.

**Failure analysis:** Profiler traces can perturb timing, so reporting a profiled duration as production latency is risky. Conversely, timing without a trace can reward an unintended fallback or cached result. Other traps include missing synchronization, timing tensor allocation, and selecting only the fastest sample.


## 10. Extend the evidence

Capture the same operation in Nsight Systems to connect CPU launch, CUDA API, and kernel timeline, then use Nsight Compute for the selected kernel's tensor-pipe and memory metrics. Repeat with autocast disabled and with an awkward shape. Build one table that keeps wall-clock effect, framework dispatch, native kernel, and hardware counters in separate columns.

The full derivation, reproduction command, evidence boundary and primary references are in [`README.md`](README.md).
